# Introduction

# Stage 04 — Stability

**Pipeline position:** fourth. Reads the row-level frame and the summary grid; writes nothing. This
is a pure reporting stage, and that is deliberate — its job is to interrogate stage 03's headline,
not to produce a new artifact.

## What this stage answers

**Does the result hold anywhere other than the window it was measured on?**

A signal that appears only on one two-season panel is not a signal. Four widenings, in increasing
order of how uncomfortable they are:

1. **Pooled panels** — 2024–2025, 2023–2025, 2021–2025. Note these are **strictly nested** (the
   library asserts it), so they are not three independent pieces of evidence. Where they disagree,
   the five-season panel carries the most information and the two-season panel the least.
2. **Per season** — the same cell computed one season at a time. This is where a signal that is
   really a two-good-years artifact reveals itself. Watch for any Wilson interval containing 0.50.
3. **Per position** — QB, RB, TE, WR separately.
4. **Veteran vs rookie, and buy vs fade** — different feature sets and opposite call directions.

## Small-cell discipline

Every cell with `n < 10` is flagged `TOO_SMALL` and carries **no directional conclusion**. This is
not a hedge but a hard rule, applied by `summarise_cell` in the shared library rather than by
whichever notebook happens to print the table. On the drafted board at high thresholds, most
per-season cells will carry that flag — and saying so is the honest output, not a failure.

## Inputs and outputs

| Direction | Path |
|---|---|
| in | `artifacts/player_season_results.csv`, `artifacts/threshold_summary.csv`, `00_shared_pipeline.ipynb` |
| out | none — reporting only |

### Explain — load the shared library

Every stage notebook begins here. It loads `00_shared_pipeline.ipynb` using the repo's convention
(`memory/prefer-ipynb-not-py.md`, mirroring the loader in `betting/predict_totals.ipynb` cell 4):
**json + exec over the library's code cells**, never `%run` (brittle across nbclient / papermill /
VSCode) and never a `.py` module (the repo is notebook-centric by rule).

`RUN_TESTS = False` and `SHARED_VERBOSE = False` are set **before** the exec, so the library's inline
tests are skipped and its configuration banner stays silent — those belong to a standalone run of the
library, not to every consumer.

The cell prints a compact load record: the SHA-256 of the library notebook itself, the count of names
imported, and the pinned parameters. Recording the library's hash means each stage's output states
exactly which version of the shared code produced it — if the library changes, the stages' recorded
hashes diverge and the mismatch is visible rather than silent.

In [1]:
import json as _json
from pathlib import Path as _Path


def _exec_notebook(path, glob):
    """Execute every code cell of a notebook into `glob` (repo convention: json + exec)."""
    with open(path, encoding="utf-8") as _fh:
        _nb = _json.load(_fh)
    for _cell in _nb["cells"]:
        if _cell["cell_type"] == "code":
            exec("".join(_cell["source"]), glob)


RUN_TESTS = False          # skip the library's inline self-tests in a consumer
SHARED_VERBOSE = False     # suppress the library's configuration banner
_SHARED = "00_shared_pipeline.ipynb"
_before = set(globals())
_exec_notebook(_SHARED, globals())
_loaded = sorted(n for n in set(globals()) - _before
                 if not n.startswith("_") and n not in {"RUN_TESTS", "SHARED_VERBOSE"})

print(f"loaded {_SHARED}")
print(f"  library sha256 : {sha256_file(_SHARED)}")
print(f"  names imported : {len(_loaded)}")
print(f"  functions      : {[n for n in _loaded if callable(globals()[n])]}")
print(f"  repo           : {REPO.name}   project: {PROJECT.name}")
print(f"  seasons {TEST_SEASONS} | thresholds {THRESHOLDS} | populations {list(POPULATIONS)}")
print(f"  seed {SEED} | perms {N_PERM:,} | boots {N_BOOT:,}")

loaded 00_shared_pipeline.ipynb
  library sha256 : d3e28a60fab75caf19c5387de293057de842c21c3c088edbc204acd89972b469
  names imported : 48
  functions      : ['Path', 'add_signals', 'boot_index_matrix', 'bootstrap_lift', 'build_ranks', 'canonical_strata', 'correct_vec', 'datetime', 'logistic_design', 'logistic_newton', 'norm', 'perm_sign_matrix', 'permutation_test', 'population_slice', 'sha256_file', 'spearmanr', 'summarise_cell', 'thr_col', 'timezone', 'wilson']
  repo           : JoSchoAnalytics   project: adp_consensus_agreement_2026-08-02
  seasons [2021, 2022, 2023, 2024, 2025] | thresholds [0.0, 5.0, 7.5, 10.0] | populations ['all_adp', 'drafted_top180']
  seed 20260802 | perms 10,000 | boots 10,000


### Interpretation — library loaded, this stage is anchored to it

The load record confirms the shared library executed cleanly and lists the names now in scope,
including the analysis functions this stage calls. The pinned parameters match the study's
declaration — seasons 2021–2025, thresholds `[0, 5, 7.5, 10]`, both populations, seed 20260802 — so
this notebook cannot silently disagree with its siblings about what a rank is or how a hit rate is
scored.

The **library SHA-256 is printed and recorded**. Every stage prints the same digest, which is what
makes "all seven stages ran against the same library" a checkable claim rather than an assumption;
stage 07 re-hashes the library and compares.

`RUN_TESTS=False` means the library's self-tests did not run here — they belong to a standalone run
of `00_shared_pipeline.ipynb`, which is the gate for this pipeline being trustworthy at all.

**This stage reads:** `artifacts/player_season_results.csv` and `artifacts/threshold_summary.csv`
**and writes:** nothing — this stage reports only

### Explain — load the results and widen the lens: pooled panels and per season

Reads stage 02's row-level frame and stage 03's summary grid, then asks whether the headline holds
outside its window.

**Pooled panels** first — 2024–2025, 2023–2025, 2021–2025 — side by side for both populations. The
shared library asserts these are **strictly nested**, so they are not three independent pieces of
evidence. Where they disagree, the five-season panel carries the most information and the two-season
panel the least, because the latter is a subset of the former.

**Then per season**, the same agreement cell computed one season at a time. This is where a signal
that is really a two-good-years artifact reveals itself. Two flags are printed for every cell: a
`TOO_SMALL` marker when `n < 10` (no directional conclusion), and an explicit note when the Wilson
interval **contains 0.50** — the season did not demonstrably beat a coin flip.

Both flags come from the shared library's `summarise_cell`, so they are applied by the scorer rather
than by whichever notebook prints the table.

In [2]:
PLAYER_RESULTS = pd.read_csv(ARTIFACTS / "player_season_results.csv")
SIGNALS = PLAYER_RESULTS.rename(columns={"position": "pos", "model_pred": "pred",
                                         "actual_half_ppr": "y", "model_family": "model"})
SUMMARY = pd.read_csv(ARTIFACTS / "threshold_summary.csv")
SUMMARY = SUMMARY[SUMMARY.market == "sleeper_adp"]
print(f"loaded {len(SIGNALS):,} row-level records and {len(SUMMARY):,} Sleeper-ADP summary cells\n")

print("=" * 112)
print("1. POOLED PANELS — agreement hit rate (universe A). NOTE: these panels are strictly NESTED.")
print("=" * 112)
_p = SUMMARY[(SUMMARY.universe == "A") & (SUMMARY.split == "all") & SUMMARY.panel.isin(POOLED_PANELS)]
print(_p.pivot_table(index=["population", "panel"], columns="threshold",
                     values=["hit_rate", "n"], aggfunc="first").round(4).to_string())

print("\n" + "=" * 112)
print("2. PER SEASON — drafted board, universe A")
print("=" * 112)
for t in THRESHOLDS:
    sub = SUMMARY[(SUMMARY.population == "drafted_top180") & (SUMMARY.universe == "A")
                  & (SUMMARY.split == "all") & SUMMARY.panel.str.startswith("season_")
                  & (SUMMARY.threshold == t)].sort_values("panel")
    print(f"\n  threshold t>{t:g}")
    for _, r in sub.iterrows():
        flag = "  <-- TOO_SMALL (n<10)" if r.too_small_n_lt_10 else ""
        straddle = ("  [interval contains 0.50]"
                    if (r.n >= 10 and r.wilson_lo <= 0.5 <= r.wilson_hi) else "")
        print(f"    {r.panel}: n={int(r.n):>3} hits={int(r.hits):>3} "
              f"hit_rate={r.hit_rate:.4f} [{r.wilson_lo:.3f}, {r.wilson_hi:.3f}]{flag}{straddle}")

_n_small = int(SUMMARY[(SUMMARY.population == "drafted_top180") & (SUMMARY.universe == "A")
                       & (SUMMARY.split == "all") & SUMMARY.panel.str.startswith("season_")
                       ].too_small_n_lt_10.sum())
print(f"\n  per-season drafted cells flagged TOO_SMALL: {_n_small} of 20")

loaded 5,315 row-level records and 1,298 Sleeper-ADP summary cells

1. POOLED PANELS — agreement hit rate (universe A). NOTE: these panels are strictly NESTED.
                                hit_rate                            n               
threshold                           0.0     5.0     7.5     10.0 0.0  5.0  7.5  10.0
population     panel                                                                
all_adp        pooled_2021_2025   0.7681  0.8566  0.8815  0.9124  940  516  422  331
               pooled_2023_2025   0.7802  0.8801  0.9024  0.9253  655  392  338  281
               pooled_2024_2025   0.7988  0.8876  0.9023  0.9228  512  338  307  259
drafted_top180 pooled_2021_2025   0.7220  0.8381  0.8857  0.8919  410  105   70   37
               pooled_2023_2025   0.6722  0.8644  0.9250  1.0000  241   59   40   19
               pooled_2024_2025   0.6605  0.8500  0.9310  1.0000  162   40   29   15

2. PER SEASON — drafted board, universe A

  threshold t>0
    season_2021

### Interpretation — the drafted-board signal is not stable, and 2025 is its weakest season

**Pooled panels behave differently in the two populations.** On the full population the rate barely
moves as the window widens (76.8 / 78.0 / 79.9% at t>0 for 2021–25 / 2023–25 / 2024–25), unsurprising
given the undrafted tail is present throughout. On the drafted board the pattern **inverts with
threshold**: at t>0 the five-season panel is the *strongest* (72.2%, n=410) and the two-season the
weakest (66.1%, n=162); at t>10 the five-season panel is the *weakest* (89.2%, n=37) while the short
panels read 100% on 19 and 15 calls. Because the panels are nested, those short-panel highs are not
corroboration — they are the same data sliced smaller.

**Per season is where the honest reading hardens.** Drafted board at t>0: 2021 **83.5%**, 2022 75.6%,
2023 69.6%, 2024 74.4%, and **2025 56.6% with a Wilson interval of [0.454, 0.671] that contains
0.50**. The most recent complete season is the only one where the threshold-0 signal fails to separate
from a coin flip, and the trend across five seasons runs downward. A signal strongest in the oldest
season and weakest in the newest is the opposite of what a durable edge looks like.

Above t>0 the per-season cells are simply too thin. At t>7.5 they give 7, 23, 11, 23 and 6 calls, with
2021 and 2025 flagged `TOO_SMALL` at 100% on 7 and 6; at t>10 they give 4, 14, 4, 12 and 3. Six of the
twenty per-season drafted cells carry the flag, and one unflagged cell (2022 at t>10, 71.4%) has an
interval containing 0.50. **No per-season claim above t>0 is supportable on the drafted board.**

### Explain — split by position, player group, and call direction

Three more ways the headline could turn out to be one subgroup wearing a trench coat.

**By position.** QB, RB, TE and WR differ enormously in how many players carry a meaningful ADP and
how noisy season totals are. If the aggregate is really one position, this shows it.

**Veteran vs rookie.** Rookies have no NFL prior, so the model scores them from an entirely different
feature set — the rookie arms use college, combine and draft-capital features. If the signal lived
only in one group, the two rates would separate.

**Buy vs fade.** "Ranked above his price" and "ranked below his price" are different claims about a
market, and there is no reason they must be equally reliable.

All three are printed for both populations at t>0 and t>5, with the `TOO_SMALL` flag carried through.
At t>5 on the drafted board several splits will have very few calls — that is a finding about the
scale of this signal, not a gap in the analysis, and the flags say so explicitly.

In [3]:
def show_split(split, panel, pop_name, thresholds=(0.0, 5.0), universe="A"):
    t = SUMMARY[(SUMMARY.panel == panel) & (SUMMARY.population == pop_name)
                & (SUMMARY.universe == universe) & (SUMMARY.split == split)
                & (SUMMARY.threshold.isin(thresholds))]
    cols = ["threshold", "split_value", "n", "hits", "misses", "ties", "hit_rate",
            "wilson_lo", "wilson_hi", "too_small_n_lt_10"]
    out = t.sort_values(["threshold", "split_value"])[cols].round(4).copy()
    out["flag"] = np.where(out.too_small_n_lt_10, "TOO_SMALL", "")
    return out.drop(columns=["too_small_n_lt_10"])


print("=" * 112)
print("3. PER POSITION — pooled 2024-2025, universe A")
print("=" * 112)
for pop_name in POPULATIONS:
    print(f"\n  -- {pop_name} --")
    print(show_split("position", "pooled_2024_2025", pop_name).to_string(index=False))

print("\n" + "=" * 112)
print("4. VETERAN vs ROOKIE, and BUY vs FADE — pooled 2024-2025, universe A")
print("=" * 112)
for pop_name in POPULATIONS:
    print(f"\n  -- {pop_name}: veteran / rookie --")
    print(show_split("group", "pooled_2024_2025", pop_name).to_string(index=False))
    print(f"  -- {pop_name}: buy / fade --")
    print(show_split("direction", "pooled_2024_2025", pop_name).to_string(index=False))

_d5 = SUMMARY[(SUMMARY.population == "drafted_top180") & (SUMMARY.universe == "A")
              & (SUMMARY.panel == "pooled_2024_2025") & (SUMMARY.threshold == 5.0)
              & (SUMMARY.split == "position")]
print(f"\n  positions represented on the drafted board at t>5: {sorted(_d5.split_value)} "
      f"(n = {dict(zip(_d5.split_value, _d5.n.astype(int)))})")
print(f"  calls per season on the drafted board: t>5 -> "
      f"{int(SUMMARY[(SUMMARY.population=='drafted_top180')&(SUMMARY.universe=='A')&(SUMMARY.panel=='pooled_2024_2025')&(SUMMARY.threshold==5.0)&(SUMMARY.split=='all')].n.iloc[0])/2:.0f}"
      f", t>10 -> "
      f"{int(SUMMARY[(SUMMARY.population=='drafted_top180')&(SUMMARY.universe=='A')&(SUMMARY.panel=='pooled_2024_2025')&(SUMMARY.threshold==10.0)&(SUMMARY.split=='all')].n.iloc[0])/2:.1f}")

3. PER POSITION — pooled 2024-2025, universe A

  -- all_adp --
 threshold split_value   n  hits  misses  ties  hit_rate  wilson_lo  wilson_hi flag
       0.0          QB  68    52      14     2    0.7647     0.6514     0.8497     
       0.0          RB 143   111      31     1    0.7762     0.7012     0.8368     
       0.0          TE 105    90      13     2    0.8571     0.7776     0.9115     
       0.0          WR 196   156      39     1    0.7959     0.7341     0.8464     
       5.0          QB  33    30       3     0    0.9091     0.7643     0.9686     
       5.0          RB  86    72      14     0    0.8372     0.7451     0.9005     
       5.0          TE  71    68       3     0    0.9577     0.8830     0.9855     
       5.0          WR 148   130      17     1    0.8784     0.8159     0.9217     

  -- drafted_top180 --
 threshold split_value  n  hits  misses  ties  hit_rate  wilson_lo  wilson_hi      flag
       0.0          QB 20    12       6     2    0.6000     0.3866  

### Interpretation — nothing separates by position, group, or direction

**By position**, drafted board at t>0, everything sits between 60% and 73% with heavily overlapping
intervals — QB 60.0% (n=20), TE 60.9% (n=23), WR 62.5% (n=56), RB 73.0% (n=63). No position separates
from the others. At t>5 the drafted board yields **2 QB calls, 20 RB, 18 WR and zero TE**, so only RB
(90.0%) has enough calls to look at, and one position on one two-season panel is no basis for a
positional claim. **TE contributing no drafted calls at all at t>5** is itself informative: it is the
position with the weakest Sleeper coverage (44.9% in the raw files), so it can rarely express a
two-sided agreement in the first place.

**Veteran vs rookie is flat** — 66.4% vs 63.2% at t>0 — so nothing here is driven by the rookie arms,
despite those arms using a completely different feature set. That is mildly reassuring: a signal that
lived only in rookies would more likely be a draft-capital artifact than a market insight.

**Buy vs fade differs at t>0** (70.0% vs 61.1% on the drafted board) but **reverses at t>5** (81.8%
buy vs 88.9% fade). With intervals this wide and an ordering that flips between thresholds, direction
is not a reliable discriminator at these sample sizes, and no directional claim should be made.

**The number to carry into stage 05** is the last line: about **20 agreement calls per season** at t>5
on the drafted board, and **7.5** at t>10. Every inferential result in the next notebook has to be
read against that scale — a lift measured on 40 calls is a different kind of evidence than one
measured on 400.

# Conclusion and Next Steps

## What this stage established

**The drafted-board signal is not stable, and its weakest season is the most recent one.**

Per season at t>0: **83.5% (2021), 75.6% (2022), 69.6% (2023), 74.4% (2024), 56.6% (2025)** — and
2025's Wilson interval **[0.454, 0.671] contains 0.50**. The trend runs downward across five seasons.
A signal that was strongest in the oldest season and weakest in the newest is the opposite of what a
durable edge looks like.

**Pooled panels invert with threshold.** At t>0 the five-season panel is the strongest (72.2%,
n=410) and the two-season the weakest (66.1%, n=162); at t>10 the five-season panel is the weakest
(89.2%, n=37) while the short panels read 100% on 19 and 15 calls. Because the panels are nested, the
short-panel highs are not corroboration — they are the same data, sliced smaller.

**Above t>0, per-season cells are too thin to read.** At t>7.5 the five seasons give 7, 23, 11, 23 and
6 calls, with 2021 and 2025 flagged `TOO_SMALL` at 100% on 7 and 6 calls. At t>10 they give 4, 14, 4,
12 and 3 — three of five flagged, and one of the unflagged (2022, 71.4%) has an interval containing
0.50. **No per-season claim above t>0 is supportable on the drafted board.**

**Nothing separates by position, group, or direction.** Drafted board at t>0, every position sits
between 60.0% and 73.0% with heavily overlapping intervals. At t>5 the drafted board yields 2 QB
calls, 20 RB, 18 WR and **zero TE** — only RB has enough to look at. Veteran vs rookie is flat (66.4%
vs 63.2%). Buy vs fade differs at t>0 (70.0% vs 61.1%) but **reverses** at t>5 (81.8% vs 88.9%), so
direction is not a reliable discriminator at these sample sizes.

## The number to carry forward

The drafted board produces about **20 agreement calls per season** at t>5 and fewer than **8** at
t>10. Every claim in the remaining stages has to be read against that scale.

## Next step

Run **`05_inference.ipynb`**. Stability is one question; whether the cell beats chance, and whether
our model adds anything to Sleeper, is another — and the second of those decides the study.